# Spatio-Temporal Momentum

Trains a single-layer PyTorch model with volatility scaling and a Sharpe-based loss on real Yahoo Finance prices. The notebook uses an editable six-stock universe because the original automatic exchange listing came from a separate provider. Install PyTorch in the notebook environment to run this strategy. The sample uses complete price observations for the shared dates.


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from datetime import datetime, timedelta
from _market_data import download_ohlcv
from typing import List, Dict, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

class Config:
    """Configure symbols, dates, and chronological model settings."""
    TICKERS = ('AAPL', 'MSFT', 'NVDA', 'AMZN', 'JPM', 'XOM')
    START_DATE = '2015-01-01'
    END_DATE = datetime.now().strftime('%Y-%m-%d')
    MOMENTUM_LOOKBACKS = [1, 20, 63, 126, 252]
    MACD_SHORT_SCALES = [8, 16, 32]
    MACD_LONG_SCALES = [24, 48, 96]
    VOLATILITY_WINDOW = 60
    TAU = 20
    VOL_TARGET = 0.15
    TRAIN_RATIO = 0.7
    VAL_RATIO = 0.15
    TEST_RATIO = 0.15
    BATCH_SIZE = 32
    EPOCHS = 100
    LEARNING_RATE = 0.001
    L1_ALPHA = 0.001
    PATIENCE = 10
    TRADING_DAYS_YEAR = 252

class YahooDataLoader:

    def get_historical_prices(self, ticker, start_date, end_date):
        try:
            prices = download_ohlcv(ticker, start_date, end_date, auto_adjust=True)
        except RuntimeError as error:
            print(error)
            return None
        return prices[['Close']].rename(columns={'Close': ticker})

    def build_price_matrix(self, tickers: List[str], start_date: str, end_date: str, min_history_days: int=400) -> pd.DataFrame:
        """Combine real price histories and use complete-case observations."""
        print(f'\nDownloading prices for {len(tickers)} symbols...')
        price_dfs = []
        successful_tickers = []
        for i, ticker in enumerate(tickers):
            if i % 50 == 0:
                print(f'Progress: {i}/{len(tickers)}')
            df = self.get_historical_prices(ticker, start_date, end_date)
            if df is not None and len(df) >= min_history_days:
                price_dfs.append(df)
                successful_tickers.append(ticker)
        if not price_dfs:
            raise ValueError('No symbol had enough observations')
        prices = pd.concat(price_dfs, axis=1, join='outer')
        prices = prices.sort_index()
        threshold = 0.95
        prices = prices.dropna(thresh=int(threshold * len(prices)), axis=1)
        prices = prices.dropna()
        print(f'\nDownloaded: {len(prices)} days, {len(prices.columns)} assets')
        print(f'Actual dates: {prices.index[0]} to {prices.index[-1]}')
        return prices

class FeatureEngineering:

    @staticmethod
    def compute_returns_volatility(prices: pd.DataFrame, vol_window: int=60) -> Tuple[pd.DataFrame, pd.DataFrame]:
        returns = prices.pct_change()
        volatility = returns.ewm(span=vol_window, min_periods=vol_window).std()
        volatility = volatility.shift(1)
        return (returns, volatility)

    @staticmethod
    def compute_momentum_features(prices: pd.DataFrame, returns: pd.DataFrame, volatility: pd.DataFrame, lookback_periods: List[int]) -> pd.DataFrame:
        features = {}
        for k in lookback_periods:
            if k > len(prices):
                continue
            cum_returns = (1 + returns).rolling(window=k, min_periods=k).apply(lambda x: np.prod(x) - 1, raw=True)
            vol_scaled = volatility * np.sqrt(k)
            vol_scaled = vol_scaled.replace(0, np.nan)
            norm_returns = cum_returns / vol_scaled
            norm_returns = norm_returns.clip(-10, 10)
            features[f'norm_return_{k}'] = norm_returns.shift(1)
        result = pd.concat(features, axis=1)
        if len(prices.columns) == 1:
            result.columns = [col[0] if isinstance(col, tuple) else col for col in result.columns]
        return result

    @staticmethod
    def compute_macd_features(prices: pd.DataFrame, short_scales: List[int], long_scales: List[int]) -> pd.DataFrame:
        features = {}
        price_std = prices.rolling(window=63, min_periods=63).std()
        price_std = price_std.replace(0, np.nan)
        for S in short_scales:
            for L in long_scales:
                if S >= L:
                    continue
                ema_short = prices.ewm(span=S, min_periods=S).mean()
                ema_long = prices.ewm(span=L, min_periods=L).mean()
                macd = (ema_short - ema_long) / price_std
                macd_std = macd.rolling(window=252, min_periods=126).std()
                macd_std = macd_std.replace(0, np.nan)
                macd_norm = macd / macd_std
                macd_norm = macd_norm.clip(-10, 10)
                features[f'macd_{S}_{L}'] = macd_norm.shift(1)
        result = pd.concat(features, axis=1)
        if len(prices.columns) == 1:
            result.columns = [col[0] if isinstance(col, tuple) else col for col in result.columns]
        return result

    @staticmethod
    def build_feature_matrix(prices: pd.DataFrame, config: Config) -> Tuple[Dict[str, pd.DataFrame], pd.DataFrame, pd.DataFrame]:
        print('\nComputing features...')
        returns, volatility = FeatureEngineering.compute_returns_volatility(prices, config.VOLATILITY_WINDOW)
        print(f'\nRaw data diagnostics:')
        print(f'  Returns - NaN: {returns.isna().sum().sum()}, Infinite: {np.isinf(returns).sum().sum()}')
        print(f'  Volatility - NaN: {volatility.isna().sum().sum()}, Infinite: {np.isinf(volatility).sum().sum()}')
        print(f'  Volatility - Min: {volatility.min().min():.6f}, Max: {volatility.max().max():.6f}')
        feature_dict = {}
        for asset in prices.columns:
            asset_prices = prices[[asset]]
            asset_returns = returns[[asset]]
            asset_vol = volatility[[asset]]
            momentum_features = FeatureEngineering.compute_momentum_features(asset_prices, asset_returns, asset_vol, config.MOMENTUM_LOOKBACKS)
            macd_features = FeatureEngineering.compute_macd_features(asset_prices, config.MACD_SHORT_SCALES, config.MACD_LONG_SCALES)
            asset_features = pd.concat([momentum_features, macd_features], axis=1)
            feature_dict[asset] = asset_features
        valid_indices = [df.dropna().index for df in feature_dict.values()]
        common_index = valid_indices[0]
        for idx in valid_indices[1:]:
            common_index = common_index.intersection(idx)
        for asset in feature_dict:
            feature_dict[asset] = feature_dict[asset].loc[common_index]
        returns = returns.loc[common_index]
        volatility = volatility.loc[common_index]
        print(f'\nFeature diagnostics:')
        sample_asset = list(feature_dict.keys())[0]
        sample_features = feature_dict[sample_asset]
        print(f'  Features per asset: {sample_features.shape[1]}')
        print(f'  Features - NaN: {sample_features.isna().sum().sum()}')
        print(f'  Features - Infinite: {np.isinf(sample_features).sum().sum()}')
        print(f'  Feature range: [{sample_features.min().min():.2f}, {sample_features.max().max():.2f}]')
        print(f'\nAssets: {len(feature_dict)}')
        print(f'Complete feature dates: {len(common_index)}')
        return (feature_dict, returns, volatility)

class STM_SLP(nn.Module):
    """Single-layer perceptron for spatio-temporal momentum."""

    def __init__(self, n_assets: int, tau: int, n_features: int):
        super(STM_SLP, self).__init__()
        input_dim = n_assets * tau * n_features
        self.fc = nn.Linear(input_dim, n_assets)
        self.activation = nn.Tanh()
        nn.init.xavier_uniform_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size = x.shape[0]
        x_flat = x.view(batch_size, -1)
        x_flat = (x_flat - x_flat.mean(dim=1, keepdim=True)) / (x_flat.std(dim=1, keepdim=True) + 1e-08)
        output = self.fc(x_flat)
        signals = self.activation(output)
        return signals

class STMDataset(Dataset):
    """Build time-ordered input windows and target returns."""

    def __init__(self, feature_dict: Dict[str, pd.DataFrame], returns: pd.DataFrame, volatilities: pd.DataFrame, tau: int):
        self.tau = tau
        self.assets = list(feature_dict.keys())
        self.n_assets = len(self.assets)
        self.n_features = feature_dict[self.assets[0]].shape[1]
        self.features_list = []
        self.returns_list = []
        self.vols_list = []
        all_dates = feature_dict[self.assets[0]].index
        valid_dates = all_dates[tau:]
        for i in range(len(valid_dates)):
            date = valid_dates[i]
            if date not in returns.index:
                continue
            date_idx = all_dates.get_loc(date)
            start_idx = date_idx - tau
            end_idx = date_idx
            if start_idx < 0:
                continue
            features_window = []
            returns_day = []
            vols_day = []
            for asset in self.assets:
                asset_features = feature_dict[asset].iloc[start_idx:end_idx].values
                asset_features = np.nan_to_num(asset_features, nan=0.0, posinf=10.0, neginf=-10.0)
                asset_features = np.clip(asset_features, -10, 10)
                features_window.append(asset_features)
                ret = returns.loc[date, asset] if asset in returns.columns else 0.0
                vol = volatilities.loc[date, asset] if asset in volatilities.columns else 0.01
                ret = 0.0 if np.isnan(ret) or np.isinf(ret) else np.clip(ret, -0.5, 0.5)
                vol = 0.01 if np.isnan(vol) or np.isinf(vol) or vol <= 0 else min(vol, 1.0)
                returns_day.append(ret)
                vols_day.append(vol)
            self.features_list.append(np.stack(features_window))
            self.returns_list.append(np.array(returns_day))
            self.vols_list.append(np.array(vols_day))
        print(f'Dataset built: {len(self.features_list)} complete samples')

    def __len__(self) -> int:
        return len(self.features_list)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        return (torch.FloatTensor(self.features_list[idx]), torch.FloatTensor(self.returns_list[idx]), torch.FloatTensor(self.vols_list[idx]))

def sharpe_loss(signals: torch.Tensor, returns: torch.Tensor, volatilities: torch.Tensor, vol_target: float=0.15) -> torch.Tensor:
    vol_scale = torch.clamp(volatilities, min=1e-06)
    scaled_returns = signals * (vol_target / vol_scale) * returns
    portfolio_returns = scaled_returns.mean(dim=1)
    portfolio_returns = torch.clamp(portfolio_returns, -0.5, 0.5)
    mean_return = portfolio_returns.mean() * Config.TRADING_DAYS_YEAR
    std_return = portfolio_returns.std() * np.sqrt(Config.TRADING_DAYS_YEAR)
    std_return = torch.clamp(std_return, min=1e-06)
    sharpe = mean_return / std_return
    if torch.isnan(sharpe) or torch.isinf(sharpe):
        return torch.tensor(0.0, requires_grad=True)
    return -sharpe

def sharpe_loss_with_l1(signals: torch.Tensor, returns: torch.Tensor, volatilities: torch.Tensor, model: nn.Module, alpha: float=0.001, vol_target: float=0.15) -> torch.Tensor:
    sharpe = sharpe_loss(signals, returns, volatilities, vol_target)
    l1_penalty = sum((torch.abs(param).sum() for param in model.parameters()))
    total_loss = sharpe + alpha * l1_penalty
    if torch.isnan(total_loss) or torch.isinf(total_loss):
        return torch.tensor(0.0, requires_grad=True)
    return total_loss

class STMTrainer:

    def __init__(self, model: nn.Module, config: Config):
        self.model = model
        self.config = config
        self.optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE)
        self.best_val_sharpe = -np.inf
        self.patience_counter = 0
        self.model_saved = False

    def train_epoch(self, train_loader: DataLoader) -> float:
        self.model.train()
        losses = []
        for features, returns, vols in train_loader:
            if torch.isnan(features).any() or torch.isnan(returns).any() or torch.isnan(vols).any():
                continue
            self.optimizer.zero_grad()
            signals = self.model(features)
            if torch.isnan(signals).any():
                print('  ⚠️  Skipping to batch with NaN signals')
                continue
            loss = sharpe_loss_with_l1(signals, returns, vols, self.model, self.config.L1_ALPHA, self.config.VOL_TARGET)
            if torch.isnan(loss) or torch.isinf(loss):
                print(f'  ⚠️  Invalid loss: {loss.item()}, saltando batch')
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()
            losses.append(loss.item())
        return np.mean(losses) if losses else float('nan')

    def evaluate(self, loader: DataLoader) -> float:
        self.model.eval()
        all_returns = []
        with torch.no_grad():
            for features, returns, vols in loader:
                if torch.isnan(features).any() or torch.isnan(returns).any() or torch.isnan(vols).any():
                    continue
                signals = self.model(features)
                if torch.isnan(signals).any():
                    continue
                scaled_returns = signals * (self.config.VOL_TARGET / (vols + 1e-08)) * returns
                portfolio_returns = scaled_returns.mean(dim=1)
                valid_returns = portfolio_returns[~torch.isnan(portfolio_returns)]
                if len(valid_returns) > 0:
                    all_returns.extend(valid_returns.numpy())
        if len(all_returns) == 0:
            return float('nan')
        all_returns = np.array(all_returns)
        sharpe = all_returns.mean() * self.config.TRADING_DAYS_YEAR / (all_returns.std() * np.sqrt(self.config.TRADING_DAYS_YEAR) + 1e-08)
        return sharpe

    def train(self, train_loader: DataLoader, val_loader: DataLoader) -> nn.Module:
        print('\nStarting training...')
        for epoch in range(self.config.EPOCHS):
            train_loss = self.train_epoch(train_loader)
            val_sharpe = self.evaluate(val_loader)
            if np.isnan(train_loss) or np.isnan(val_sharpe):
                print(f'Epoch {epoch + 1}/{self.config.EPOCHS} - Train Loss: {train_loss:.4f} - Val Sharpe: {val_sharpe:.4f} ⚠️')
            else:
                print(f'Epoch {epoch + 1}/{self.config.EPOCHS} - Train Loss: {train_loss:.4f} - Val Sharpe: {val_sharpe:.4f}')
            if not np.isnan(val_sharpe) and val_sharpe > self.best_val_sharpe:
                self.best_val_sharpe = val_sharpe
                self.patience_counter = 0
                torch.save(self.model.state_dict(), 'best_stm_model.pth')
                self.model_saved = True
                print(f'  → Saved best model (Sharpe: {val_sharpe:.4f})')
            else:
                self.patience_counter += 1
                if self.patience_counter >= self.config.PATIENCE:
                    print(f'\nEarly stopping at epoch {epoch + 1}')
                    break
        if self.model_saved:
            self.model.load_state_dict(torch.load('best_stm_model.pth', weights_only=True))
            print(f'\nLoaded model with validation Sharpe: {self.best_val_sharpe:.4f}')
        else:
            raise RuntimeError('No model reached to finite validation score.')
        return self.model

def main():
    """Train, validate, and evaluate the momentum model."""
    config = Config()
    print('=' * 60)
    print('PHASE 1: PRICE DOWNLOAD')
    print('=' * 60)
    loader = YahooDataLoader()
    tickers = config.TICKERS
    if not tickers:
        raise ValueError('Set one or more symbols in Config.TICKERS')
    prices = loader.build_price_matrix(tickers, config.START_DATE, config.END_DATE)
    print('\n' + '=' * 60)
    print('PHASE 2: FEATURE ENGINEERING')
    print('=' * 60)
    feature_dict, returns, volatility = FeatureEngineering.build_feature_matrix(prices, config)
    print('\n' + '=' * 60)
    print('PHASE 3: DATASET PREPARATION')
    print('=' * 60)
    full_dataset = STMDataset(feature_dict, returns, volatility, config.TAU)
    n_samples = len(full_dataset)
    train_end = int(n_samples * config.TRAIN_RATIO)
    val_end = train_end + int(n_samples * config.VAL_RATIO)
    if not 0 < train_end < val_end < n_samples:
        raise ValueError('The requested history is too short for training, validation, and test splits.')
    train_dataset = torch.utils.data.Subset(full_dataset, range(0, train_end))
    val_dataset = torch.utils.data.Subset(full_dataset, range(train_end, val_end))
    test_dataset = torch.utils.data.Subset(full_dataset, range(val_end, len(full_dataset)))
    train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=False)
    val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False)
    print(f'Train samples: {len(train_dataset)}')
    print(f'Val samples: {len(val_dataset)}')
    print(f'Test samples: {len(test_dataset)}')
    print('\n' + '=' * 60)
    print('PHASE 4: MODEL TRAINING')
    print('=' * 60)
    n_assets = len(prices.columns)
    n_features = full_dataset.n_features
    model = STM_SLP(n_assets, config.TAU, n_features)
    print(f'\nModel: {n_assets} assets, {config.TAU} lookback days, {n_features} features')
    print(f'Total parameters: {sum((p.numel() for p in model.parameters())):,}')
    trainer = STMTrainer(model, config)
    model = trainer.train(train_loader, val_loader)
    print('\n' + '=' * 60)
    print('PHASE 5: TEST EVALUATION')
    print('=' * 60)
    train_sharpe = trainer.evaluate(train_loader)
    val_sharpe = trainer.evaluate(val_loader)
    test_sharpe = trainer.evaluate(test_loader)
    print(f'\nFinal results:')
    print(f'  Train Sharpe: {train_sharpe:.4f}')
    print(f'  Val Sharpe: {val_sharpe:.4f}')
    print(f'  Test Sharpe: {test_sharpe:.4f}')
    return (model, prices, feature_dict, returns, volatility)
if __name__ == '__main__':
    model, prices, feature_dict, returns, volatility = main()
